# Локальная диаризация аудио с pyannote.audio

### Установка зависимостей
После первой установки выбрать Среда выполнения -> Перезапустить сеанс, затем продолжить с ячейки инициализации ниже, не запуская установку повторно.

In [1]:
%pip install -q "pyannote.audio==4.0.4" pandas python-dotenv soundfile memory_profiler


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Инициализация девайса и библиотек
После перезапуска начать отсюда.


In [1]:
import json
import os
import subprocess
from getpass import getpass
from pathlib import Path
from time import perf_counter

import pandas as pd
from memory_profiler import memory_usage
import soundfile as sf
import torch
from IPython.display import display

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU доступен: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден. Будет использован CPU.")


GPU не найден. Будет использован CPU.


### Настройка токена Hugging Face

Создать аккаунт Hugging Face, принять условия модели [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1), создать токен с правом чтения и добавить его в секреты под именем HF_TOKEN.

In [2]:
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
if not HF_TOKEN:
    raise RuntimeError("Токен Hugging Face не введён.")
print("Токен получен.")

Токен получен.


### Выбор локального аудиофайла

Указать путь к одному локальному аудиофайлу или медиаконтейнеру с аудиодорожкой в переменной AUDIO_PATH.

In [4]:
AUDIO_PATH = Path(r"content/audio_files/EN2001e_300sec.wav")
audio_path = AUDIO_PATH.expanduser().resolve()
audio_name = audio_path.name
allowed_extensions = {
    ".3g2", ".3gp", ".aac", ".ac3", ".aif", ".aifc", ".aiff",
    ".amr", ".ape", ".au", ".avi", ".awb", ".caf", ".dts",
    ".eac3", ".flac", ".flv", ".gsm", ".m2ts", ".m4a", ".m4b",
    ".mka", ".mkv", ".mov", ".mp2", ".mp3", ".mp4", ".mpc",
    ".mpeg", ".mpg", ".mts", ".oga", ".ogg", ".opus", ".ra",
    ".rm", ".snd", ".spx", ".tak", ".ts", ".tta", ".wav",
    ".wave", ".webm", ".wma", ".wv",
}
if audio_path.suffix.lower() not in allowed_extensions:
    raise ValueError(
        f"Неподдерживаемый формат {audio_path.suffix or 'без расширения'}. "
        "Выберите аудиофайл или медиаконтейнер с поддерживаемой аудиодорожкой."
    )
if not audio_path.is_file() or audio_path.stat().st_size == 0:
    raise ValueError(f"Локальный файл отсутствует или пуст: {audio_path}")
print(f"Выбран файл: {audio_path} ({audio_path.stat().st_size / 1024 / 1024:.2f} МБ)")

Выбран файл: D:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\content\audio_files\EN2001e_300sec.wav (9.16 МБ)


### Подготовка аудио

In [5]:
prepared_dir = Path("content/prepared_audio")
prepared_dir.mkdir(parents=True, exist_ok=True)
prepared_path = prepared_dir / f"{audio_path.stem}_mono_16khz.wav"
command = [
    "ffmpeg", "-v", "error", "-y", "-i", str(audio_path),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(prepared_path),
]
try:
    conversion = subprocess.run(command, capture_output=True, text=True, check=False)
except FileNotFoundError as exc:
    raise RuntimeError("FFmpeg not found.") from exc

if conversion.returncode != 0 or not prepared_path.is_file() or prepared_path.stat().st_size <= 44:
    details = (conversion.stderr or "FFmpeg error").strip()[-1000:]
    raise RuntimeError(
        "Не удалось прочитать или преобразовать аудио."
        f"Сообщение FFmpeg: {details}"
    )
print(f"Аудио подготовлено: {prepared_path} (mono, 16 кГц, WAV)")

Аудио подготовлено: content\prepared_audio\EN2001e_300sec_mono_16khz.wav (mono, 16 кГц, WAV)


### Загрузка модели

In [6]:
from pyannote.audio import Pipeline

MODEL_ID = "pyannote/speaker-diarization-community-1"
try:
    pipeline = Pipeline.from_pretrained(MODEL_ID, token=HF_TOKEN)
    if pipeline is None:
        raise RuntimeError("Pipeline.from_pretrained returned None.")
    pipeline.to(DEVICE)
except Exception as exc:
    message = str(exc)
    raise RuntimeError(f"Не удалось загрузить модель: {message}") from exc
print(f"Модель загружена на {DEVICE}.")

d:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only th

Модель загружена на cpu.


### Выполнение диаризации

При известности числа участников, задать значение NUM_SPEAKERS целым положительным числом. Значение None включает автоматическое определение.

In [7]:
NUM_SPEAKERS = None

if NUM_SPEAKERS is not None and (isinstance(NUM_SPEAKERS, bool) or not isinstance(NUM_SPEAKERS, int) or NUM_SPEAKERS < 1):
    raise ValueError("NUM_SPEAKERS должен быть положительным целым числом или None.")

inference_options = {} if NUM_SPEAKERS is None else {"num_speakers": NUM_SPEAKERS}
audio_samples, sample_rate = sf.read(
    prepared_path,
    dtype="float32",
    always_2d=True,
)
audio_input = {
    "waveform": torch.from_numpy(audio_samples.T),
    "sample_rate": sample_rate,
}
def run_diarization():
    return pipeline(audio_input, **inference_options)

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
diarization_started_at = perf_counter()
try:
    if DEVICE.type == "cuda":
        output = run_diarization()
    else:
        memory_values, output = memory_usage(
            run_diarization,
            retval=True,
            interval=0.1,
        )
except (torch.cuda.OutOfMemoryError, MemoryError) as exc:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise RuntimeError(
        "Во время диаризации закончилась оперативная или видеопамять. "
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Диаризация завершилась ошибкой: {exc}."
    ) from exc
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    used_memory_gib = torch.cuda.max_memory_allocated() / 1024**3
    memory_measurement_name = "Пиковое использование VRAM"
else:
    used_memory_gib = max(memory_values) / 1024
    memory_measurement_name = "Пиковое использование RAM"
diarization_elapsed_seconds = perf_counter() - diarization_started_at

segments = [
    {"start": round(float(turn.start), 3), "end": round(float(turn.end), 3), "speaker": str(speaker)}
    for turn, speaker in output.speaker_diarization
]
if not segments:
    raise RuntimeError(
        "В записи не обнаружена речь."
    )
print(f"Диаризация завершена. Найдено сегментов: {len(segments)}")


Диаризация завершена. Найдено сегментов: 113


### Просмотр и сохранение временной разметки

Сегменты выводятся в таблице в минутах, а временная разметка сохраняется локально в JSON (в секундах).

In [8]:
def format_minutes(seconds):
    minutes, remaining_seconds = divmod(float(seconds), 60)
    return f"{int(minutes):02d}:{remaining_seconds:06.3f}"

table = pd.DataFrame(
    [
        (format_minutes(item["start"]), format_minutes(item["end"]), item["speaker"])
        for item in segments
    ],
    columns=["Начало", "Окончание", "Говорящий"],
)
print(
    f"Время диаризации: {format_minutes(diarization_elapsed_seconds)} "
    f"({diarization_elapsed_seconds:.3f} с)"
)
print(f"{memory_measurement_name}: {used_memory_gib:.3f} GiB")
with pd.option_context("display.max_rows", None):
    display(table)

result = {"audio_file": audio_name, "segments": segments}
results_dir = Path("content/diarization_json_files/local/pyannotate/speaker-diarization-community-1")
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / Path(audio_name).with_suffix(".json").name
with result_path.open("w", encoding="utf-8") as file:
    json.dump(result, file, ensure_ascii=False, indent=2)

print(f"Результат сохранён: {result_path.resolve()}")


Время диаризации: 06:53.710 (413.710 с)
Пиковое использование RAM: 1.820 GiB


,Начало,Окончание,Говорящий
0,00:00.031,00:04.064,SPEAKER_02
1,00:01.027,00:01.111,SPEAKER_03
2,00:01.887,00:03.018,SPEAKER_03
3,00:03.035,00:10.223,SPEAKER_03
4,00:08.924,00:09.295,SPEAKER_02
5,00:09.295,00:09.346,SPEAKER_04
6,00:09.346,00:09.397,SPEAKER_02
7,00:09.397,00:11.793,SPEAKER_04
8,00:11.793,00:12.670,SPEAKER_03
9,00:12.367,00:12.738,SPEAKER_04


Результат сохранён: D:\ЛЭТИ магистратура\диплом\практика_весенний_семестр\program\content\diarization_json_files\local\pyannotate\speaker-diarization-community-1\EN2001e_300sec.json
